# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print top-level metadata
print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
# Show keywords/usecases
print("Keywords:", getattr(metadata, "keywords", "N/A"))
print("Example use-cases:", getattr(metadata, "dataUseCases", "N/A"))

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we discover the record sets present in the dataset, their `@id`s, and their fields. All entities are referenced by their `@id`, as required for FAIRML/MlCroissant interoperability.

In [ ]:
# Get all record sets.
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    # Some Croissant schemas use 'hasPart' or 'dataset' to refer to record sets
    record_sets = getattr(metadata, 'hasPart', [])

if not record_sets:
    raise RuntimeError("No record sets found in this Croissant schema.")

print(f"Found {len(record_sets)} record sets.\n")

overview = []
for i, rs in enumerate(record_sets):
    print(f"Record set {i+1}:")
    print(f"  @id  : {rs['@id']}")
    print(f"  name : {rs.get('name','--')}")
    print(f"  Description: {rs.get('description', '--')}")
    fields = rs.get('field', [])
    if fields and isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields in this record set:")
    for f in fields:
        print(f"    - {f['@id']}: {f.get('name','--')} ({f.get('dataType','--')})")
    overview.append({'record_set_id': rs['@id'], 'fields': fields})

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

> For reproducibility, adjust the variable values below using the overview info above. By default, the first record set is selected.

In [ ]:
# For demonstration, use the first record set identified above
record_sets_ids = [entry['record_set_id'] for entry in overview]
dataframes = {}

for rs_id in record_sets_ids:
    print(f"Loading record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")
        continue
    df = pd.DataFrame(records)
    print(f"  Loaded {len(df)} records, columns: {list(df.columns)}")
    dataframes[rs_id] = df

# For downstream processing, pick the first loaded record set (with data)
for rs_id, df in dataframes.items():
    if len(df):
        main_record_set = rs_id
        main_df = df
        break
else:
    print("No records loaded successfully; check schema/croissant definition.")
    main_record_set = None

if main_record_set:
    print(f"\nColumns in main record set ({main_record_set}):\n{main_df.columns.tolist()}")
    print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We will select a numeric column and perform sample processing. Replace the IDs with the actual field `@id`s for your dataset if necessary.

In [ ]:
# Example: Try to pick a numeric field - let's guess 'mw' (Molecular Weight) or 'count_peptides'
numeric_candidates = [col for col in main_df.columns if main_df[col].dtype.kind in 'fi']

if not numeric_candidates:
    # try to parse columns to numeric
    for col in main_df.columns:
        try:
            pd.to_numeric(main_df[col].dropna().iloc[0])
            numeric_candidates.append(col)
        except Exception:
            continue

# pick the first numeric candidate
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    raise RuntimeError("No numeric field found for demonstration.")

print(f"Using numeric field: {numeric_field}")

threshold = main_df[numeric_field].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field]) else 10

# Filter records based on the threshold
filtered_df = main_df.copy()
filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field], errors='coerce') > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df[[numeric_field]].head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by another likely categorical field
categorical_candidates = [col for col in main_df.columns if main_df[col] != numeric_field and main_df[col].nunique()<20]
if categorical_candidates:
    group_field = categorical_candidates[0]
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field} on filtered records:")
    print(grouped_df.head())
else:
    group_field = None
    print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we show a histogram of the selected numeric field, and a box plot grouped by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
main_df[numeric_field].astype(float).hist(bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.grid(True)
plt.show()

# Boxplot by group field, if present
if group_field:
    plt.figure(figsize=(8, 4))
    main_df.boxplot(column=numeric_field, by=group_field, grid=False)
    plt.title(f"{numeric_field} by {group_field}")
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset directly from a Croissant schema, explored its metadata, reviewed available record sets and fields (referenced by their `@id`), and performed basic exploratory data analysis and visualization.

- All dataset entities were referenced using their `@id` per FAIRML/MLCommons recommendations.
- Data loading, filtering, normalization, and grouping steps were demonstrated.
- Visualization of distributions and grouped statistics was shown.

For detailed or domain-specific analysis, consult the official dataset description, accompanying documentation, and metadata fields. You may adapt or extend the code using specific field `@id`s as needed for your project.
